Step 1: Bootstrap and install instructions

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython
!pip install --quiet langgraph
from google.colab import userdata, drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')


import importlib.metadata
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 1. Check version
try:
    print(f"✓ langgraph {importlib.metadata.version('langgraph')}")
except importlib.metadata.PackageNotFoundError:
    raise RuntimeError("langgraph not installed — run !pip install langgraph")

# 2. Build and run a minimal graph to verify imports and execution
class _State(TypedDict):
    status: str

def _node(state: _State) -> _State:
    return {"status": "ok"}

_g = StateGraph(_State)
_g.add_node("test", _node)
_g.add_edge(START, "test")
_g.add_edge("test", END)

result = _g.compile().invoke({"status": "start"})

if result.get("status") != "ok":
    raise AssertionError(f"Expected status 'ok', got {result}")

print("✓ Graph compiled and executed successfully!")

Step 2: Milestone 4 having complex graph

In [ ]:
from astra_swarm.graph import graph_triage, triage_graph
from astra_swarm.correlation import correlate
from astra_swarm.cassette import cassette
from IPython.core.display import Image, display
import json
from pathlib import Path
from pydantic import BaseModel

# Show the compiled graph — visually more complex than Week 3
display(Image(triage_graph.get_graph().draw_mermaid_png()))

alerts = json.loads(
    Path("/content/astra-swarm/data/synthetic/02_alerts.json").read_text()
)

# Run the full graph
with cassette("week4_milestone"):
    incidents = [graph_triage(a) for a in alerts]

# Correlate (correlation.py still works — TypedDict names changed but structure is compatible)
correlated = correlate(incidents)

# Persist
out = Path("/content/astra-swarm/data/synthetic/10_incidents.json")
#out.write_text(json.dumps([
#    {**{k: v.model_dump() if hasattr(v, 'model_dump') else v for k, v in inc.items()}}
#    for inc in incidents
#], indent=2, default=str))
out.write_text(json.dumps([
    {k: v.model_dump() if isinstance(v, BaseModel) else v for k, v in inc.items()}
    for inc in incidents
], indent=2, default=str))

Step 3: Metrics

In [ ]:
from collections import Counter

# Supervisor decision distribution — what actions did supervisors take?
all_decisions = [
    d["next_action"]
    for inc in incidents
    for d in inc.get("supervisor_decisions", [])
]
print(f"Supervisor decisions: {dict(Counter(all_decisions))}")

# Average supervisor calls per incident
avg_calls = sum(inc.get("supervisor_call_count", 0) for inc in incidents) / len(incidents)
print(f"Avg supervisor calls per incident: {avg_calls:.1f}")

# Forced terminations (supervisor cap hit)
forced = sum(
    1 for inc in incidents
    for d in inc.get("supervisor_decisions", [])
    if d.get("forced")
)
print(f"Forced terminations (cap hit): {forced}")

# Worker call distribution
worker_calls = Counter(w for inc in incidents for w in inc.get("workers_run", []))
print(f"Worker call counts: {dict(worker_calls)}")

# ITDR specialist quality
itdr_run = sum(1 for inc in incidents if inc.get("identity") is not None)
itdr_findings = sum(
    1 for inc in incidents
    if (ident := inc.get("identity")) and any([
        ident.impossible_travel,
        ident.mfa_fatigue,
        ident.privilege_escalation,
        ident.dormant_reactivation,
    ])
)
print(f"ITDR: ran on {itdr_run}/10 incidents; found threats in {itdr_findings}")

# Escalations
escalated = sum(1 for inc in incidents if inc.get("escalated"))
print(f"Escalated: {escalated}/10")

# Correlation
print(f"Correlation: 10 incidents → {len(correlated)} correlated groups")